# HeliosLM own base (0.3B) — pretraining on a free Kaggle TPU v5e-8

Plan and budget: `docs/OWN_BASE_0P3B.md`.

**Before running (once):**
1. Kaggle account with a verified phone number (TPU access needs it).
2. *Settings → Accelerator → TPU v5e-8*, *Internet → On*.
3. *Add-ons → Secrets*: add `HF_TOKEN` (a Hugging Face **write** token) and tick it for this notebook.

**Every session:** *Run all*. The run resumes from the last checkpoint in the private
checkpoint repo, trains for up to 8.3 hours, saves and uploads, then stops before Kaggle's
9-hour session limit. Repeat next session until the log shows the final step; the last
session also exports a Llama-format model and uploads it.

In [ ]:
# ---- settings (edit only these) ----
HF_USER   = "chienhsinlin"
TOKENS    = f"{HF_USER}/helioslm-own-0p3b-tokens"   # private dataset (token shards)
CKPT_REPO = f"{HF_USER}/helioslm-own-0p3b-ckpt"     # private model repo (resume checkpoints)
OUT_REPO  = f"{HF_USER}/helioslm-own-0p3b"          # private model repo (final Llama-format export)
BRANCH    = "main"
TRAIN_TOKENS = "6e9"     # 6B first; raise to 2e10 later (same CKPT_REPO keeps going only if unchanged)

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
!pip -q install -U huggingface_hub safetensors ml_dtypes
import jax; print(jax.__version__, jax.devices())

In [ ]:
!rm -rf helioslm && git clone -q --depth 1 -b {BRANCH} https://github.com/tonythetiger168/helioslm
from huggingface_hub import hf_hub_download
for f in ("mix_main.json", "mix_decay.json", "tokenizer.json"):
    hf_hub_download(TOKENS, f, repo_type="dataset", local_dir="/kaggle/working/data")
!ls -la /kaggle/working/data

In [ ]:
!cd /kaggle/working && python helioslm/examples/jax_pretrain_tpu.py \
    --preset 0.3b --mix data/mix_main.json --mix-decay data/mix_decay.json \
    --tokens {TRAIN_TOKENS} --batch 64 --grad-accum 4 --seq 2048 --lr 3e-3 \
    --log-every 20 --eval-every 1000 --save-every 2000 --max-hours 8.3 \
    --out /kaggle/working/run --hf-ckpt-repo {CKPT_REPO} \
    --export /kaggle/working/export --tokenizer data/tokenizer.json

In [ ]:
# runs only after the last session (export exists)
import os
from huggingface_hub import HfApi
if os.path.exists("/kaggle/working/export/config.json"):
    api = HfApi()
    api.create_repo(OUT_REPO, private=True, exist_ok=True)
    api.upload_folder(repo_id=OUT_REPO, folder_path="/kaggle/working/export")
    print("uploaded", OUT_REPO)
else:
    print("not finished yet: run this notebook again next session")
!tail -3 /kaggle/working/run/log.jsonl